# LightGBM LambdaRank · diario de experimentos (Adri · B)

**Qué es:** un `LGBMRanker` que puntúa **cada tarea del snapshot** a partir de sus columnas (y de features que la comparan
con el resto de tareas del mismo snapshot). Dentro de cada snapshot se ordenan las tareas por puntuación y las 3 primeras
son la predicción. Es el modelo de referencia del equipo y el que mejor funciona.

**Cómo usar este notebook**
1. Ejecuta *Preparación* una vez (carga datos y define features y funciones, ~30 s).
2. Cada bloque tiene: **hipótesis** → **variantes** (se cambia UNA cosa respecto a la base del bloque) → **tabla** → **conclusión**.
3. Los bloques ya hechos (1, 2, 4, 4b, 5) tienen la conclusión escrita y **no se repiten** salvo que pongas `REPETIR_HECHOS = True`;
   el 6 (versión final) solo se lanza con `GENERAR_FINAL = True`. Así un *Run All* solo ejecuta lo nuevo.
4. Lo que gane se apunta en `MEJOR` (bloque 5) y el bloque 6 genera la versión para el stacking.

**Reglas que respeta el modelo** (`CLAUDE.md` / `EQUIPO.md`)
- Cada snapshot se predice **solo con sus propias filas**: toda feature nueva se calcula con `groupby("sample_id")`
  (o con grupos más finos dentro del snapshot: avión, tipo).
- CV con `Data/folds.csv` (5 folds por controlador; cada sesión queda entera en un fold).
- No usa `sample_id, session, atco, scenario, timestamp, task_name, row_index` como features
  (`task_name` solo se usa para sacar el avión y agrupar).

**Cómo leer las cifras**
- `cv` = top-3 hit rate out-of-fold sobre train+validation (13 175 snapshots).
- `peor_atco` = nota del peor controlador. Importa: el leaderboard privado es **un solo controlador** (023).
- Con 1 semilla, diferencias < ~0.005 son ruido; con 2 semillas, < ~0.003. Veredicto: **mejora** si Δ ≥ `MIN_GAIN`.
- Cada experimento se apunta en `experiments/lgb_runs.csv` (celda *Historial* al final).

> **Resumen (20/10, 22:30):** versión en `main` = **`lgb_adri` · base + `alerta` + `contexto` · CV 0.656** (3 semillas),
> peor controlador 0.571. Con varias semillas **empata con la base** (0.657): el LightGBM limpio parece en su techo.
> Lo que no funciona: meter todos los grupos a la vez, `exit_coord`, `deadline` (bloques 1–2), `avion_tipos`, cambiar la
> truncación, `rank_xendcg` ni regularizar (bloque 4). **Análisis de errores (bloque 3):** el 83 % de los fallos es por elegir
> **el avión equivocado**; exit_coordination (9 %) y entry_coordination (39 %) son los que más fallos acumulan.
> **21/10, 00:20:** el re-ranker (bloque 11) empeora; **limpiar las etiquetas ruidosas** (bloque 12) sube a **0.665** con R=10
> (+0.8, por encima del stacking de Juanma, 0.660). Pendiente: confirmarlo sin fuga (12b), ideas de Pablo (12c) y la versión
> final `lgb_adri_limpio` (12d). Sin ejecutar todavía: reglas `logic_*` (7) y modelos alternativos (8–9).

## Preparación

### Configuración
`PARAMS` son los parámetros base de `EQUIPO.md`. Cada experimento cambia **una** clave respecto a ellos.

In [ ]:
import json, subprocess, sys, time
import numpy as np, pandas as pd, lightgbm as lgb
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "Data"
for p in ("oof", "preds", "submissions", "models", "experiments"):
    (ROOT / p).mkdir(exist_ok=True)
for p in ("src", "cv"):
    sys.path.insert(0, str(ROOT / p))
from features import add_logic          # reglas logic_* de Juanma (solo usan el snapshot)
from metric import save_outputs         # formato común oof/preds/submissions

NAME = "lgb_adri"   # nombre de los archivos de salida
TYPES = ['entry_conflict_resolution', 'exit_conflict_resolution', 'entry_coordination', 'exit_coordination',
         'assume_aircraft', 'transfer_aircraft', 'conflict_resolution', 'non_conformance_resolution',
         'qos_improvement', 'entry_conditions', 'exit_conditions', 'aircraft_request', 'zone_conflict',
         'return_to_route']

PARAMS = dict(objective="lambdarank", n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=50,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, lambdarank_truncation_level=5,
              random_state=0, verbose=-1)
SEEDS_BLOQUE = [0, 1]        # semillas de cada variante en los bloques nuevos (se promedian las predicciones)
SEEDS_FINAL = [0, 1, 2]      # semillas de la versión para el stacking
MIN_GAIN = 0.003             # Δ mínimo para decir "mejora" con 2 semillas
REPETIR_HECHOS = False       # True = volver a lanzar los bloques ya hechos (1, 2, 4, 4b, 5; ~20 min)
GENERAR_FINAL = False        # True = el bloque 6 vuelve a generar oof/preds/modelo de lgb_adri (~3 min)

### Datos
- `lab` = train + validation (con etiqueta), cada fila con su `fold` (0–4) de `Data/folds.csv`.
- `test` = test, sin etiqueta.
- `cs` = callsign del avión de la tarea (sacado de `task_name`). **No es una feature**: solo sirve para agrupar las tareas
  de un mismo avión.

In [ ]:
def load(name):
    d = pd.read_csv(DATA / f"{name}.csv")
    pref = d.task_index.map(lambda t: len(TYPES[t]) + 1)
    d["cs"] = [n[p:].split("_")[0] for n, p in zip(d.task_name, pref)]
    return d

lab = pd.concat([load("train"), load("validation")], ignore_index=True)
lab = lab.merge(pd.read_csv(DATA / "folds.csv"), on="atco", how="left")
test = load("test")
lab = lab.sort_values(["sample_id", "row_index"]).reset_index(drop=True)
test = test.sort_values(["sample_id", "row_index"]).reset_index(drop=True)
ATCO = lab.drop_duplicates("sample_id").set_index("sample_id").atco

assert lab.fold.notna().all(), "hay controladores sin fold"
assert (lab.groupby("sample_id").target_index.sum() == 1).all(), "alguna muestra no tiene exactamente 1 respuesta"
print(f"lab: {lab.sample_id.nunique():,} muestras, {len(lab):,} filas | test: {test.sample_id.nunique():,} muestras")

### Features
Cada grupo es una función que recibe el dataframe y devuelve **solo las columnas nuevas**. Todas agrupan por `sample_id`
(o por avión / tipo dentro del snapshot), así que solo usan el propio snapshot.

| Grupo | Origen | Qué añade |
|---|---|---|
| `snapshot` | EQUIPO.md | Ranking y diferencia respecto al mínimo **dentro del snapshot** de columnas de urgencia/actividad |
| `tipo` | EQUIPO.md | Los mismos rankings **dentro de las tareas del mismo tipo** |
| `avion` | EQUIPO.md | Si la tarea es la más antigua de su avión, cuántos tipos distintos tiene el avión |
| `tipo_estado` | EQUIPO.md + Pau #2 | Cruces tipo × estado: assume con oferta, transfer debido, entry_coordination con gemelo (`ts_twin_entry` = `is_entry_losing` de Pau)… |
| `contexto` | EQUIPO.md | Cómo está el snapshot: nº de aviones, transfers/assumes/zonas/conflictos pendientes |
| `exit_coord` | Pau #3 | exit_coordination × coordinación con el sector siguiente (`k_out_*`, mismatch, herramienta), gemelo con transfer, edad frente a la tarea más nueva del avión |
| `alerta` | Pau #4 | Conflictos con alerta STCA en pantalla, nº de alertas del snapshot, ranking del tiempo al mínimo entre las tareas de conflicto |
| `deadline` | Adri | Un "tiempo límite" común a todos los tipos y su ranking en el snapshot y en el tipo |
| `avion_tipos` | bloque 4 | **Qué otras tareas tiene abiertas el mismo avión** (assume, transfer, exit_coord…), nº de tareas del avión y la más antigua |
| `logic` | Juanma (`src/features.py`) | 15 reglas `logic_*`: plazo por tipo, si la tarea es **factible** (no se dan instrucciones a un avión no asumido), si está en su **ventana típica**, orden *earliest-deadline-first*… |

Pau #5 (ranking de distancia al borde dentro del tipo) ya es una columna original (`l_rank_absf1_type`).
Pau #1 (oráculo) se descarta: usa `scenario` y `timestamp`, prohibidos como features.

In [ ]:
META = ['sample_id', 'target_index', 'session', 'atco', 'scenario', 'timestamp', 'task_name', 'row_index',
        'fold', 'cs']
BASE = [c for c in lab.columns if c not in META]
RANK_COLS = ["a_since_flight", "x_task_age", "x_state_age", "time_to_MD", "k_timeToExitPoint",
             "distance_to_zone", "abs_tts"]
ENTRY_TYPES, EXIT_TYPES, CONFLICT_TYPES = [0, 2, 4, 9], [1, 3, 5, 10], [0, 1, 6]


def _with_abs(df):
    return df.assign(abs_tts=df.time_to_sector.abs())


def _ac_any(df, mask):
    """1 si alguna tarea del mismo avión (en el mismo snapshot) cumple mask."""
    return mask.astype(int).groupby([df.sample_id, df.cs]).transform("max")


def fe_snapshot(df):
    x = _with_abs(df)
    g = x.groupby("sample_id")
    out = {}
    for c in RANK_COLS:
        out[f"rs_{c}"] = g[c].rank(method="min")
        out[f"ds_{c}"] = x[c] - g[c].transform("min")
    return pd.DataFrame(out, index=df.index)


def fe_tipo(df):
    x = _with_abs(df)
    g = x.groupby(["sample_id", "task_index"])
    out = {}
    for c in RANK_COLS:
        out[f"rt_{c}"] = g[c].rank(method="min")
        out[f"dt_{c}"] = x[c] - g[c].transform("min")
    return pd.DataFrame(out, index=df.index)


def fe_avion(df):
    g = df.groupby(["sample_id", "cs"])
    out = {
        "ac_age_rank": g.x_task_age.rank(method="min", ascending=False),   # 1 = la tarea más antigua del avión
        "ac_n_types": g.task_index.transform("nunique"),
        "ac_has_assume": _ac_any(df, df.task_index == 4),
    }
    return pd.DataFrame(out, index=df.index)


def fe_tipo_estado(df):
    t = df.task_index
    out = {
        "ts_assume_offered": ((t == 4) & (df.x_transfer_in == 1)).astype(int),
        "ts_transfer_due": ((t == 5) & (df.k_aircraftNeedsToBeTransferred == 1)).astype(int),
        "ts_entry_coord_done": ((t == 2) & (df.x_isc_coordinated == 1)).astype(int),
        "ts_entry_notified": ((t == 2) & (df.x_isc_notified == 1)).astype(int),
        "ts_zone_dist": df.distance_to_zone.where(t == 12),
        "ts_conflict_ttmd": df.time_to_MD.where(t.isin(CONFLICT_TYPES)),
        "ts_twin_assume": ((t == 4) & (df.l_twin_AE == 1)).astype(int),
        "ts_twin_entry": ((t == 2) & (df.l_twin_AE == 1)).astype(int),     # = is_entry_losing de Pau
    }
    return pd.DataFrame(out, index=df.index)


def fe_contexto(df):
    g = df.groupby("sample_id")
    tmp = pd.DataFrame({
        "sample_id": df.sample_id,
        "is_assume_offer": ((df.task_index == 4) & (df.x_transfer_in == 1)).astype(int),
        "is_transfer_due": ((df.task_index == 5) & (df.k_aircraftNeedsToBeTransferred == 1)).astype(int),
        "is_zone": (df.task_index == 12).astype(int),
        "is_conflict": df.task_index.isin(CONFLICT_TYPES).astype(int),
        "is_entry_coord": (df.task_index == 2).astype(int),
    })
    gt = tmp.groupby("sample_id")
    out = {"cx_n_aircraft": g.cs.transform("nunique")}
    for c in tmp.columns[1:]:
        out["cx_n_" + c[3:]] = gt[c].transform("sum")
    return pd.DataFrame(out, index=df.index)


def fe_exit_coord(df):
    t = df.task_index
    ec = t == 3
    out = {
        "ec_notified": (ec & (df.k_out_NOTIFIED == 1)).astype(int),
        "ec_coordinated": (ec & (df.k_out_COORDINATED == 1)).astype(int),
        "ec_mismatch": (ec & (df.k_outCoordMismatch == 1)).astype(int),
        "ec_tool": (ec & (df.k_coordinationToolActivation == 1)).astype(int),
        "ec_revision": df.k_outCoordRevision.where(ec),
        "ec_with_transfer": (ec & (_ac_any(df, t == 5) == 1)).astype(int),
        "tr_with_exit_coord": ((t == 5) & (_ac_any(df, ec) == 1)).astype(int),
        "age_vs_newest_ac": df.x_task_age - df.groupby(["sample_id", "cs"]).x_task_age.transform("min"),
    }
    return pd.DataFrame(out, index=df.index)


def fe_alerta(df):
    conf = df.task_index.isin(CONFLICT_TYPES)
    stca = df.HMI_conflict_safety_alert == 1
    ttmd = df.time_to_MD.where(conf)
    out = {
        "al_conflict_stca": (conf & stca).astype(int),
        "al_n_stca": stca.astype(int).groupby(df.sample_id).transform("sum"),
        "al_ttmd_rank": ttmd.groupby(df.sample_id).rank(method="min"),
        "al_conflict_multi": (conf & (df.multiple_conflicts_ind == 1)).astype(int),
        "al_conflict_dmd": df.distance_to_MD.where(conf),
    }
    return pd.DataFrame(out, index=df.index)


def fe_deadline(df):
    t = df.task_index
    dl = pd.Series(np.nan, index=df.index)
    entry, exit_, zone = t.isin(ENTRY_TYPES), t.isin(EXIT_TYPES), t == 12
    dl[entry] = df.time_to_sector[entry]
    dl[exit_] = df.k_timeToExitPoint.fillna(df.time_to_sector.abs())[exit_]
    dl[t == 6] = df.time_to_MD[t == 6]
    dl[zone] = (df.distance_to_zone * 1852 / df.k_groundSpeed.fillna(230))[zone]   # NM -> m, m/s
    out = {
        "dl": dl,
        "dl_rank": dl.groupby(df.sample_id).rank(method="min"),
        "dl_rank_type": dl.groupby([df.sample_id, t]).rank(method="min"),
        "dl_minus_min": dl - dl.groupby(df.sample_id).transform("min"),
    }
    return pd.DataFrame(out, index=df.index)


def fe_avion_tipos(df):
    g = df.groupby(["sample_id", "cs"])
    out = {}
    for t, nm in [(2, "entry_coord"), (3, "exit_coord"), (4, "assume"), (5, "transfer"), (10, "exit_cond"),
                  (12, "zone"), (13, "return_route"), (6, "conflict")]:
        out[f"av_has_{nm}"] = _ac_any(df, df.task_index == t)
    out["av_n_tasks"] = g.task_index.transform("size")
    out["av_max_age"] = g.x_task_age.transform("max")
    return pd.DataFrame(out, index=df.index)


def fe_logic(df):
    x = add_logic(df)
    return x[[c for c in x.columns if c.startswith("logic_")]]


FEATURE_GROUPS = {"snapshot": fe_snapshot, "tipo": fe_tipo, "avion": fe_avion,
                  "tipo_estado": fe_tipo_estado, "contexto": fe_contexto, "exit_coord": fe_exit_coord,
                  "alerta": fe_alerta, "deadline": fe_deadline, "avion_tipos": fe_avion_tipos, "logic": fe_logic}


def build(df, groups):
    """Devuelve (df con las columnas nuevas, lista de features a usar)."""
    new = [FEATURE_GROUPS[g](df) for g in groups]
    out = pd.concat([df] + new, axis=1)
    return out, BASE + [c for n in new for c in n.columns]

### Entrenamiento, CV y experimentos
- `fit`: entrena un `LGBMRanker`. `group` le dice qué filas forman cada snapshot, para que aprenda a **ordenar dentro del snapshot**.
- `hit3`: para cada snapshot, 1 si la respuesta está entre las 3 de mayor score.
- `cv`: para cada fold, entrena con los otros 4 y predice ese; con varias semillas promedia sus predicciones. El resultado es el **OOF**.
- `experimento`: una variante → CV, peor controlador, y la apunta en `experiments/lgb_runs.csv`.
- `lanzar`: un bloque entero → base del bloque + variantes → **tabla con Δ y veredicto**.

In [ ]:
def fit(A, F, params, w=None):
    m = lgb.LGBMRanker(**params)
    m.fit(A[F], A.target_index, group=A.groupby("sample_id", sort=False).size().values, sample_weight=w,
          categorical_feature=["task_index"])
    return m


def hit3(d, score):
    t = d[["sample_id", "target_index"]].assign(s=np.asarray(score))
    t = t.sort_values(["sample_id", "s"], ascending=[True, False])
    return t.groupby("sample_id").head(3).groupby("sample_id").target_index.max()


def cv(d, F, params, seeds=(0,)):
    oof = np.zeros(len(d))
    for f in range(5):
        tr, va = (d.fold != f).values, (d.fold == f).values
        for s in seeds:
            oof[va] += fit(d[tr], F, {**params, "random_state": s}).predict(d.loc[va, F]) / len(seeds)
    h = hit3(d, oof)
    fold_of = d.drop_duplicates("sample_id").set_index("sample_id").fold
    return oof, h, h.groupby(fold_of.reindex(h.index)).mean()


LOG = ROOT / "experiments" / "lgb_runs.csv"
OOFS = {}   # nombre -> (d, oof, aciertos por muestra), para analizar después


def experimento(nombre, groups, params=None, seeds=SEEDS_BLOQUE, nota=""):
    params = {**PARAMS, **(params or {})}
    t0 = time.time()
    d, F = build(lab, groups)
    oof, h, por_fold = cv(d, F, params, seeds)
    OOFS[nombre] = (d, oof, h)
    peor = h.groupby(ATCO.reindex(h.index)).mean().min()
    fila = dict(hora=time.strftime("%H:%M"), nombre=nombre, grupos="+".join(groups) or "-",
                params_cambiados=str({k: v for k, v in params.items() if PARAMS.get(k) != v}),
                n_features=len(F), cv_top3=round(h.mean(), 4), std_folds=round(por_fold.std(), 4),
                nota=f"{len(seeds)} semillas. peor_atco {peor:.3f}. {nota}".strip())
    pd.DataFrame([fila]).to_csv(LOG, mode="a", header=not LOG.exists(), index=False)
    print(f"{nombre:40s} CV {h.mean():.4f}  peor_atco {peor:.3f}  (folds {' '.join(f'{v:.3f}' for v in por_fold)})"
          f"  {len(F)} feats  {time.time() - t0:.0f}s", flush=True)
    return h.mean(), peor


def lanzar(bloque, variantes, grupos, params=None, seeds=SEEDS_BLOQUE):
    """variantes = [(nombre, por qué, grupos extra, params extra), ...]. Devuelve la tabla del bloque."""
    print(f"=== bloque {bloque}: base = {grupos or 'columnas originales'} {params or ''}  ({len(seeds)} semillas)")
    filas = [("base", *experimento(f"{bloque}/base", grupos, params, seeds, "referencia del bloque"))]
    for nombre, por_que, g_extra, p_extra in variantes:
        filas.append((nombre, *experimento(f"{bloque}/{nombre}", grupos + g_extra, {**(params or {}), **p_extra},
                                           seeds, por_que)))
    t = pd.DataFrame(filas, columns=["variante", "cv", "peor_atco"])
    t["Δ_vs_base"] = t.cv - t.cv.iloc[0]
    t["veredicto"] = ["" if i == 0 else "mejora" if v >= MIN_GAIN else "empeora" if v <= -MIN_GAIN else "ruido"
                      for i, v in enumerate(t["Δ_vs_base"])]
    return t.round(4)


def historial(n=20):
    return pd.read_csv(LOG).tail(n)

## 0 · Punto de partida

LightGBM LambdaRank con **las columnas originales** y los parámetros de `EQUIPO.md`, folds de `Data/folds.csv`:
**CV 0.651** con 1 semilla, **0.657** con 2 semillas (folds 0.683 / 0.659 / 0.647 / 0.640 / 0.630).

**Diagnóstico:** el modelo acierta muy bien los tipos frecuentes (transfer 86 %, assume 83 %, zone 71 %) y casi nunca
los raros (exit_coordination 10 %, exit_conditions 19 %). La nota varía mucho entre controladores (0.63–0.70 según el fold):
el controlador del privado puede caer en cualquier punto de ese rango.

## 1 · Features de `EQUIPO.md` (ronda 1)

**Hipótesis:** el árbol puntúa cada tarea por separado; darle hechas las comparaciones con el resto del snapshot
(rankings, diferencias, contexto) debería ayudarle a ordenar.

| Cambio | Por qué podría ayudar |
|---|---|
| `snapshot` | posición relativa de la tarea en el snapshot (urgencia, antigüedad, actividad reciente) |
| `tipo` | lo mismo pero solo frente a las tareas del mismo tipo (FIFO por tipo) |
| `avion` | distinguir las tareas de un mismo avión |
| `tipo_estado` | señales que solo tienen sentido para un tipo (assume con oferta pendiente, transfer debido…) |
| `contexto` | cómo de cargado está el snapshot |

1 semilla, ~5 min.

In [ ]:
if REPETIR_HECHOS:
    tabla1 = lanzar("b1", [(g, "ronda 1", [g], {}) for g in ["snapshot", "tipo", "avion", "tipo_estado", "contexto"]]
                    + [("todos", "todos juntos", ["snapshot", "tipo", "avion", "tipo_estado", "contexto"], {})],
                    [], seeds=[0])
    display(tabla1)
else:
    print("Bloque ya ejecutado: ver la conclusión. REPETIR_HECHOS = True para repetirlo.")

**Conclusión bloque 1 · mejoras pequeñas por separado, y juntas empeoran.**

| Variante | CV | Δ vs base (0.651) | |
|---|---|---|---|
| contexto | 0.655 | +0.004 | ruido / mejora justa |
| tipo_estado | 0.655 | +0.004 | ruido / mejora justa |
| tipo | 0.654 | +0.003 | ruido |
| avion | 0.653 | +0.001 | ruido |
| snapshot | 0.652 | +0.000 | ruido |
| **todos juntos** | 0.648 | **−0.004** | **empeora** |

- Con 133 columnas, muchas redundantes, el árbol se sobreajusta a los controladores de entrenamiento.
- `tipo_estado` repetido con 2 semillas da 0.6565 frente a 0.6570 de la base: **no aporta**.
- Las columnas originales `l_rank_*` ya dan buena parte de la posición relativa: por eso `snapshot` y `tipo` aportan poco.

## 2 · Ideas de Pau y `deadline` (ronda 2)

**Hipótesis:** el EDA de Pau (sección 9 de `01_eda.ipynb`) señala dónde falla el modelo; features pensadas para esos
tipos deberían subir su acierto.

| Cambio | Por qué podría ayudar |
|---|---|
| `exit_coord` (Pau #3) | exit_coordination acierta un 10 %: decirle cuándo la coordinación con el sector siguiente está pendiente o desajustada |
| `alerta` (Pau #4) | conflict_resolution acierta un 30 %: la alerta STCA en pantalla es lo que el controlador ve |
| `deadline` (Adri) | un tiempo límite comparable entre tipos distintos (entrada, salida, conflicto, zona) |
| `contexto` | el mejor de la ronda 1, para combinar |

1 semilla, selección hacia delante (cada grupo solo se queda si sube la CV), ~5 min.

In [ ]:
if REPETIR_HECHOS:
    tabla2 = lanzar("b2", [(g, "ronda 2", [g], {}) for g in ["contexto", "exit_coord", "alerta", "deadline"]]
                    + [("alerta+contexto", "combinación de los dos mejores", ["alerta", "contexto"], {})],
                    [], seeds=[0])
    display(tabla2)
else:
    print("Bloque ya ejecutado: ver la conclusión. REPETIR_HECHOS = True para repetirlo.")

**Conclusión bloque 2 · `alerta` es la única idea de Pau que funciona; con 2 semillas la mejora se diluye.**

| Variante | CV (1 semilla) | Δ vs base (0.651) | |
|---|---|---|---|
| **alerta + contexto** | **0.660** | **+0.008** | **mejora** |
| alerta | 0.657 | +0.006 | mejora |
| contexto | 0.655 | +0.004 | mejora justa |
| exit_coord | 0.651 | −0.001 | ruido |
| deadline | 0.651 | −0.000 | ruido |

- Confirmado con 2 semillas: **alerta + contexto = 0.6569**, base = 0.6570 → **empate**. Parte de la mejora con 1 semilla
  era suerte (se elige al ganador entre varios intentos ruidosos).
- Aun así cambia *en qué* acierta: entry_coordination 35 % → 39 %, conflict_resolution 32 % → 34 %, return_to_route 51 % → 53 %.
- `exit_coord` no arregla exit_coordination (sigue en ~8–10 %): las columnas `k_out_*` ya estaban en la base; el árbol ya podía usarlas.
- Es la versión que está ahora en `main` (`oof/lgb_adri.csv`).

## 3 · Análisis de errores

**Pregunta:** ¿dónde y cómo falla? Sirve para decidir qué atacar. Se mira sobre el OOF (predicciones hechas sin haber visto
a ese controlador). `analizar_errores` vale para cualquier OOF del equipo (`"nn_tf_v2"`, `"types"`…). ~20 s.

Qué mide:
1. **Avión correcto, tarea equivocada:** si el avión de la respuesta está en el top-3 pero con otra de sus tareas.
2. **Fallos por tipo correcto:** acierto, nº de fallos y qué parte del total de fallos supone cada tipo.
3. **Posición de la respuesta cuando falla:** si se queda cerca (4ª–5ª) o lejos.
4. **Qué pone el modelo en 1ª posición cuando falla.**
5. **Gemelos:** cuando el avión correcto tiene dos tareas que compiten, cuál elige el controlador.

In [ ]:
CORTO = ['entry_conflict', 'exit_conflict', 'entry_coord', 'exit_coord', 'assume', 'transfer', 'conflict', 'non_conf',
         'qos', 'entry_cond', 'exit_cond', 'request', 'zone', 'return_route']


def analizar_errores(nombre):
    d = lab.merge(pd.read_csv(ROOT / "oof" / f"{nombre}.csv"), on=["sample_id", "row_index"])
    d["rk"] = d.groupby("sample_id").score.rank(ascending=False, method="first")
    d["typ"] = d.task_index.map(lambda t: CORTO[t])
    y = d[d.target_index == 1].set_index("sample_id")
    y["hit"] = y.rk <= 3
    ac_top3 = d[d.rk <= 3].groupby("sample_id").cs.apply(set)
    y["avion_ok"] = [cs in ac_top3[s] for s, cs in zip(y.index, y.cs)]
    fallo = ~y.hit
    print(f"{nombre}: top-3 {y.hit.mean():.4f} | {fallo.sum()} fallos, de ellos con el AVIÓN correcto en el top-3: "
          f"{(fallo & y.avion_ok).sum()} ({(fallo & y.avion_ok).sum() / fallo.sum():.0%})")

    g = y.groupby("typ").agg(muestras=("hit", "size"), acierto=("hit", "mean"), fallos=("hit", lambda h: (~h).sum()))
    g["%_de_los_fallos"] = (g.fallos / fallo.sum() * 100).round(1)
    g["avion_ok_tarea_mal"] = (fallo & y.avion_ok).groupby(y.typ).sum()
    g["posición_mediana_si_falla"] = y[fallo].groupby("typ").rk.median()
    display(g.sort_values("fallos", ascending=False).round(3))

    print("Posición de la respuesta cuando falla:")
    display(pd.cut(y[fallo].rk, [3, 4, 5, 6, 10, 20, 99]).value_counts(sort=False).rename("fallos").to_frame().T)

    print("Cuando falla, tipo que el modelo pone 1º (filas = tipo correcto):")
    top1 = d[d.rk == 1].set_index("sample_id").typ
    ct = pd.crosstab(y[fallo].typ, top1.reindex(y[fallo].index))
    display(ct.loc[g.sort_values("fallos", ascending=False).index[:7]])

    print("Gemelos: el avión correcto tiene las dos tareas, ¿cuál elige el controlador?")
    tiene = d.groupby(["sample_id", "cs"]).typ.agg(set)
    filas = []
    for a, b in [("exit_coord", "transfer"), ("entry_coord", "assume"), ("return_route", "transfer"),
                 ("exit_cond", "transfer"), ("exit_coord", "exit_cond"), ("zone", "transfer")]:
        r = y[y.typ.isin([a, b])]
        r = r[[{a, b} <= tiene.get((s, cs), set()) for s, cs in zip(r.index, r.cs)]]
        filas.append(dict(par=f"{a} vs {b}", casos=len(r), elige_el_primero=round((r.typ == a).mean(), 3),
                          acierto_modelo=round(r.hit.mean(), 3)))
    display(pd.DataFrame(filas))


analizar_errores(NAME)

**Conclusión del análisis de errores (`lgb_adri`, CV 0.657):**

| Tipo correcto | Muestras | Acierto top-3 | % de todos los fallos | Posición mediana si falla |
|---|---|---|---|---|
| exit_coordination | 745 | **0.09** | **15 %** | 11ª |
| entry_coordination | 1.068 | 0.39 | **14 %** | 9ª |
| zone_conflict | 2.121 | 0.71 | 14 % | 6ª |
| assume | 3.366 | 0.84 | 12 % | 5ª |
| transfer | 3.527 | 0.86 | 11 % | 5ª |
| return_to_route | 937 | 0.50 | 10 % | 5ª |
| exit_conditions | 448 | 0.19 | 8 % | 10ª |
| conflict_resolution | 390 | 0.32 | 6 % | 6ª |

1. **El problema es el avión, no la tarea:** solo el **17 %** de los fallos tiene el avión correcto en el top-3. En el
   83 % restante el modelo está mirando **otros aviones**. Mejorar el "gemelo" (assume vs entry_coord…) tiene un techo bajo.
2. **Un tercio de los fallos es por poco:** 1.589 fallos (35 %) tienen la respuesta en 4ª–5ª posición.
3. **Juega a lo seguro:** cuando falla, pone en 1ª posición transfer, assume o zone casi siempre (como la red de Pablo).
4. **exit_coordination está enterrada** (mediana 11ª): el modelo casi nunca la sube. Cuando el avión tiene exit_coord y
   transfer, el controlador elige exit_coord solo un **11 %** de las veces: el modelo aprende "transfer gana" y hace bien
   en media, pero pierde esos casos.
5. **Gemelos:** entry_coord vs assume → el controlador elige entry_coord un 27 %; return_to_route vs transfer un 30 %;
   exit_conditions vs transfer un 16 %.

**Qué atacar:** (a) que el modelo sepa qué **otras tareas tiene el avión** (el contexto de cada avión, no de cada tarea);
(b) que la función de pérdida mire **el top-3** y no el top-5; (c) corregir el sesgo hacia los tipos frecuentes.

## 4 · Atacar los fallos

**Hipótesis:** según el bloque 3, el modelo falla sobre todo al elegir el avión y entierra los tipos raros. Se prueban
cambios dirigidos a eso **sobre la versión actual** (`alerta` + `contexto`), con 2 semillas.

| Cambio | Por qué podría ayudar |
|---|---|
| `avion_tipos` | el árbol ve cada tarea aislada; saber **qué otras tareas tiene abiertas el avión** (assume, transfer, exit_coord…) da el contexto del avión. Ataca el 83 % de fallos de avión y los gemelos |
| `truncation_level=3` | LambdaRank optimiza el orden de las 5 primeras; nuestra métrica solo mira las **3 primeras** |
| `truncation_level=10` | al revés: dar peso a respuestas que caen en 6ª–10ª (exit_coord, entry_coord) para subirlas |
| `rank_xendcg` | otra pérdida de ranking, más robusta al **ruido en las etiquetas** (los controladores no coinciden mucho) |
| `regularizado` | `num_leaves` 15, `min_child_samples` 100, `lr` 0.03 × 700 árboles: árboles más simples, menos memorizar controladores |

~1 min por variante, ~6–7 min el bloque.

In [ ]:
GRUPOS_ACTUALES = ["alerta", "contexto"]   # la versión de main (bloque 2)

if REPETIR_HECHOS:
    tabla4 = lanzar("b4", [
        ("avion_tipos",  "qué otras tareas tiene el avión",          ["avion_tipos"], {}),
        ("trunc=3",      "la pérdida mira solo el top-3",            [], dict(lambdarank_truncation_level=3)),
        ("trunc=10",     "dar peso a respuestas en 6ª-10ª",          [], dict(lambdarank_truncation_level=10)),
        ("xendcg",       "pérdida más robusta al ruido",             [], dict(objective="rank_xendcg")),
        ("regularizado", "árboles más simples, menos memorizar",     [],
         dict(num_leaves=15, min_child_samples=100, learning_rate=0.03, n_estimators=700)),
    ], GRUPOS_ACTUALES)
    display(tabla4)
else:
    print("Bloque ya ejecutado: ver la conclusión. REPETIR_HECHOS = True para repetirlo.")

**Conclusión bloque 4 · ninguna variante mejora la CV; el modelo está en su techo con este enfoque.**

| Variante (2 semillas) | CV | Δ vs base (0.6569) | peor controlador | |
|---|---|---|---|---|
| trunc=3 | 0.6556 | −0.001 | 0.567 | ruido |
| trunc=10 | 0.6537 | −0.003 | 0.576 | ruido / empeora |
| avion_tipos | 0.6532 | −0.004 | 0.568 | empeora |
| regularizado | 0.6503 | −0.007 | **0.588** | empeora (pero mejor peor controlador) |
| xendcg | 0.6421 | −0.015 | 0.570 | empeora |

- **`avion_tipos` no ayuda:** el árbol ya deduce el contexto del avión de las columnas `l_*` y `ac_*`; decírselo no añade nada.
- **La truncación no importa:** optimizar el top-3, el top-5 o el top-10 da lo mismo.
- **`regularizado`** pierde 0.7 puntos de media pero sube el peor controlador (+1.8): candidato si se prima la robustez, no la media.
- **Lectura general:** con las columnas de un solo snapshot, cambiar la pérdida, la capacidad o añadir contexto del avión
  no mueve la nota. Lo que queda es probar **otra información** (reglas `logic_*`, bloque 7) u **otros modelos** (bloque 8).

### 4b · Corregir el sesgo hacia los tipos frecuentes (post-proceso)

**Hipótesis:** el modelo juega a lo seguro (transfer, assume, zone). Sumar un pequeño **sesgo por tipo** a la puntuación
podría subir exit_coordination, exit_conditions… sin reentrenar. Los sesgos se ajustan en 4 folds y se miden en el 5º
(CV anidada), así que la nota es honesta. Se aplica sobre `oof/lgb_adri.csv`. ~1 min, no entrena nada.

In [ ]:
def sesgo_por_tipo(d, oof, grid=(-1.0, -0.5, -0.25, 0, 0.25, 0.5, 1.0), tipos=(3, 10, 2, 12, 13, 6, 8, 1, 4, 5)):
    fold, ti = d.fold.values, d.task_index.values

    def acierto(score, mask):
        return hit3(d[mask], score[mask]).mean()

    out = np.zeros(len(d))
    for f in range(5):
        trm = fold != f
        b, mejor_b = np.zeros(14), acierto(oof, trm)
        for _ in range(2):
            for t in tipos:
                for v in grid:
                    bb = b.copy(); bb[t] = v
                    s = acierto(oof + bb[ti], trm)
                    if s > mejor_b + 1e-4:
                        mejor_b, b = s, bb
        out[fold == f] = (oof + b[ti])[fold == f]
        print(f"fold {f}: sesgos " + ", ".join(f"{TYPES[t]} {v:+}" for t, v in enumerate(b) if v))
    return hit3(d, out).mean()


if REPETIR_HECHOS:
    d4 = lab.merge(pd.read_csv(ROOT / "oof" / f"{NAME}.csv"), on=["sample_id", "row_index"])
    print(f"\nsin sesgo: {hit3(d4, d4.score).mean():.4f}   con sesgo por tipo (CV anidada): "
          f"{sesgo_por_tipo(d4, d4.score.values):.4f}")
else:
    print("Bloque ya ejecutado: ver la conclusión. REPETIR_HECHOS = True para repetirlo.")

**Conclusión bloque 4b · se ejecutó, pero el resultado no quedó guardado** (el notebook no se guardó con salidas).
Si hace falta, repetir con `REPETIR_HECHOS = True` (~1 min) y apuntar aquí el antes/después.

## 5 · Combinar lo que ha funcionado

Pon en `MEJOR_GRUPOS` / `MEJOR_PARAMS` solo los cambios con veredicto **mejora**. Varios cambios que ayudan por separado no
siempre suman: la celda de comprobación los mide juntos con 2 semillas.

In [ ]:
MEJOR_GRUPOS = ["alerta", "contexto"]   # + ["logic"] si mejora en el bloque 7
MEJOR_PARAMS = {}                       # bloque 4: ningún cambio de parámetros mejora

In [ ]:
if REPETIR_HECHOS:
    cv_comb, peor_comb = experimento("b5/comb", MEJOR_GRUPOS, MEJOR_PARAMS, nota="combinación de ganadores")
    print(f"combinación: {cv_comb:.4f} (peor controlador {peor_comb:.3f})")
else:
    print("Bloque ya ejecutado: ver la conclusión. REPETIR_HECHOS = True para repetirlo.")

**Conclusión bloque 5 · no hay nada nuevo que combinar.** `MEJOR` = base + `alerta` + `contexto` con los parámetros de
`EQUIPO.md`: CV 0.6569 con 2 semillas, peor controlador 0.570 (igual que la base del bloque 4).

## 6 · Versión para el stacking

Entrena `MEJOR` con **3 semillas** y guarda en el formato común. Juanma lo usa en el stacking (**nadie más sube a Kaggle**).
~3 min. **Solo con `GENERAR_FINAL = True`** (sobrescribe los archivos de `lgb_adri`):
- `oof/lgb_adri.csv` — predicciones OOF (stacking de Juanma, análisis de errores)
- `preds/lgb_adri.csv` y `submissions/sub_lgb_adri.csv` — predicciones de test (media de las semillas)
- `models/lgb_adri_s<semilla>.txt` — **los pesos**, entrenados con todo train+validation
- `models/lgb_adri_config.json` — grupos, parámetros y features usados

In [ ]:
if GENERAR_FINAL:
    params = {**PARAMS, **MEJOR_PARAMS}
    d, F = build(lab, MEJOR_GRUPOS)
    oof, h, por_fold = cv(d, F, params, SEEDS_FINAL)
    peor = h.groupby(ATCO.reindex(h.index)).mean().min()
    print(f"CV top-3 final ({len(SEEDS_FINAL)} semillas): {h.mean():.4f}  peor controlador {peor:.3f}"
          f"  (folds: {' '.join(f'{v:.3f}' for v in por_fold)})")
    pd.DataFrame([dict(hora=time.strftime("%H:%M"), nombre=f"FINAL {NAME}", grupos="+".join(MEJOR_GRUPOS) or "-",
                       params_cambiados=str(MEJOR_PARAMS), n_features=len(F), cv_top3=round(h.mean(), 4),
                       std_folds=round(por_fold.std(), 4),
                       nota=f"{len(SEEDS_FINAL)} semillas. peor_atco {peor:.3f}. Modelo guardado")]
                 ).to_csv(LOG, mode="a", header=False, index=False)
    d[["sample_id", "row_index"]].assign(score=oof).to_csv(ROOT / "oof" / f"{NAME}.csv", index=False)

    dt, _ = build(test, MEJOR_GRUPOS)
    score = np.zeros(len(dt))
    for s in SEEDS_FINAL:
        m = fit(d, F, {**params, "random_state": s})
        m.booster_.save_model(str(ROOT / "models" / f"{NAME}_s{s}.txt"))
        score += m.predict(dt[F]) / len(SEEDS_FINAL)
    json.dump(dict(groups=MEJOR_GRUPOS, params=params, seeds=SEEDS_FINAL, features=F, cv_top3=round(h.mean(), 4)),
              open(ROOT / "models" / f"{NAME}_config.json", "w"), indent=1)

    pred = dt[["sample_id", "row_index"]].assign(score=score)
    pred.to_csv(ROOT / "preds" / f"{NAME}.csv", index=False)
    sub = (pred.sort_values(["sample_id", "score"], ascending=[True, False]).groupby("sample_id").row_index
               .apply(lambda r: " ".join(map(str, r.iloc[:3]))).rename("prediction").reset_index())
    sub.to_csv(ROOT / "submissions" / f"sub_{NAME}.csv", index=False)
    print("Guardado:", NAME, "| grupos:", MEJOR_GRUPOS, "| params:", MEJOR_PARAMS or "base", "| test:", len(sub), "muestras")
else:
    print("No se regenera lgb_adri (GENERAR_FINAL = False). La versión en main es la de la conclusión de abajo.")

In [ ]:
# Comprobación con el scorer oficial sobre las muestras de validation.csv: debe coincidir con nuestro hit3
if GENERAR_FINAL:
    val_ids = set(pd.read_csv(DATA / "validation.csv", usecols=["sample_id"]).sample_id)
    o = d[["sample_id", "row_index"]].assign(score=oof)
    o = o[o.sample_id.isin(val_ids)].sort_values(["sample_id", "score"], ascending=[True, False])
    val_sub = o.groupby("sample_id").row_index.apply(lambda r: " ".join(map(str, r.iloc[:3]))).rename("prediction")
    val_sub.reset_index().to_csv(ROOT / "submissions" / f"val_{NAME}.csv", index=False)
    print(f"hit3 propio en validation: {h[h.index.isin(val_ids)].mean():.4f}")
    r = subprocess.run([sys.executable, DATA / "scorer.py", "--labelled", DATA / "validation.csv",
                        ROOT / "submissions" / f"val_{NAME}.csv"], capture_output=True, text=True)
    print("scorer oficial:", r.stdout or r.stderr)

**Resultado de la versión para el stacking (en `main`)**

| | CV top-3 | peor controlador |
|---|---|---|
| base (columnas originales, 2 semillas) | 0.657 | 0.570 |
| **`lgb_adri`** (alerta + contexto, **3 semillas**) | **0.656** | **0.571** |

Juanma la combina con la red y los especialistas por tipo: `stack_meta` (limpio) **0.660**.

**Cómo usan los demás el modelo guardado (sin entrenar)** — necesitan las funciones de *Features* (`build`):
```python
cfg = json.load(open("models/lgb_adri_config.json"))
dt, F = build(test, cfg["groups"])
score = np.mean([lgb.Booster(model_file=f"models/lgb_adri_s{s}.txt").predict(dt[F]) for s in cfg["seeds"]], axis=0)
```

## 7 · Reglas de Juanma (`logic_*`)

**Hipótesis:** Juanma escribió en `src/features.py` 15 reglas "de controlador" que solo usan el snapshot y **nunca se han probado
en este LightGBM** (Pablo las probó con futuro, donde el futuro lo tapa todo). Las más prometedoras para nuestros fallos:
- `logic_feasible`: no se dan instrucciones a un avión que no controlas → descarta tareas imposibles ahora mismo.
- `logic_in_window` / `logic_window_dist`: si la tarea está en la **ventana de tiempo** típica en que los controladores la hacen.
- `logic_edf_rank`: orden *earliest-deadline-first* entre las tareas factibles (la regla clásica de planificación).

| Variante | Qué se añade | Por qué |
|---|---|---|
| `+logic` | las 15 reglas sobre la versión actual (alerta + contexto) | información nueva, pensada a mano |
| `base+logic` | las 15 reglas sobre las columnas originales | por si `alerta`/`contexto` y `logic` se pisan |

2 semillas, ~3 min.

In [ ]:
tabla7 = lanzar("b7", [
    ("+logic", "reglas de Juanma sobre la versión actual", ["logic"], {}),
], GRUPOS_ACTUALES)
cv_bl, peor_bl = experimento("b7/base+logic", ["logic"], nota="reglas de Juanma sobre las columnas originales")
display(tabla7)
print(f"base+logic (sin alerta/contexto): {cv_bl:.4f}  peor controlador {peor_bl:.3f}  (base sola: 0.657)")

**Conclusión bloque 7 · (rellenar tras ejecutar)**

| Variante | CV | Δ vs base | peor controlador | |
|---|---|---|---|---|
| … | | | | |

- Si `+logic` mejora: añadir `"logic"` a `MEJOR_GRUPOS` (bloque 5) y regenerar con `GENERAR_FINAL = True` (bloque 6).

## 8 · Modelos alternativos

**Hipótesis:** el LambdaRank está en su techo, pero **otro modelo puede equivocarse en sitios distintos**. Aunque saque un poco
menos, si falla en otros snapshots **suma en el stacking de Juanma** (el bloque 9 lo mide). Todos con las mismas features
(base + alerta + contexto) y los mismos folds, 1 semilla.

| Modelo | Qué cambia | Por qué podría aportar |
|---|---|---|
| `lgb_bin` | LightGBM **binario** (¿es esta la tarea? sí/no), no ranking | otra pérdida: puntúa cada tarea por sí misma, no por comparación |
| `cat_rank` | **CatBoost YetiRank** | árboles simétricos y otra forma de tratar `task_index`; suele ordenar distinto que LightGBM |
| `xgb_rank` | **XGBoost** `rank:pairwise` | otro algoritmo de árboles, pérdida por pares |

**Antes de ejecutar:** CatBoost y XGBoost no están instalados en `.venv`. En la terminal de VS Code (desde la carpeta del proyecto):
```
.venv\Scripts\python.exe -m pip install catboost xgboost
```
y reinicia el kernel (botón *Restart*), ejecuta *Preparación* otra vez y vuelve aquí. Usan la GPU (RTX 3050) si la encuentran.
Tiempos aproximados: `lgb_bin` ~1 min, `xgb_rank` ~2 min, `cat_rank` ~3–8 min (GPU) o mucho más en CPU.

In [ ]:
def cv_modelo(nombre, entrenar, grupos, seeds=(0,), guardar=False, nota=""):
    """CV con cualquier modelo. entrenar(A, F, seed) -> función que puntúa un DataFrame de features.
    guardar=True entrena también con todo y escribe oof/, preds/ y submissions/ (formato común) para el stacking."""
    t0 = time.time()
    d, F = build(lab, grupos)
    oof = np.zeros(len(d))
    for f in range(5):
        tr, va = (d.fold != f).values, (d.fold == f).values
        for s in seeds:
            oof[va] += entrenar(d[tr], F, s)(d.loc[va, F]) / len(seeds)
    h = hit3(d, oof)
    OOFS[nombre] = (d, oof, h)
    peor = h.groupby(ATCO.reindex(h.index)).mean().min()
    fila = dict(hora=time.strftime("%H:%M"), nombre=nombre, grupos="+".join(grupos) or "-", params_cambiados="otro modelo",
                n_features=len(F), cv_top3=round(h.mean(), 4), std_folds=np.nan,
                nota=f"{len(seeds)} semillas. peor_atco {peor:.3f}. {nota}".strip())
    pd.DataFrame([fila]).to_csv(LOG, mode="a", header=not LOG.exists(), index=False)
    print(f"{nombre:20s} CV {h.mean():.4f}  peor_atco {peor:.3f}  {time.time() - t0:.0f}s", flush=True)
    if guardar:
        dt, _ = build(test, grupos)
        score = np.mean([entrenar(d, F, s)(dt[F]) for s in seeds], axis=0)
        save_outputs(nombre, d[["sample_id", "row_index"]].assign(score=oof),
                     dt[["sample_id", "row_index"]].assign(score=score))
    return h.mean()


def ent_lgb_bin(A, F, s):
    p = {k: v for k, v in PARAMS.items() if k not in ("objective", "lambdarank_truncation_level")}
    m = lgb.LGBMClassifier(**{**p, "random_state": s}, objective="binary")
    m.fit(A[F], A.target_index, categorical_feature=["task_index"])
    return lambda X: m.predict_proba(X)[:, 1]


def ent_cat(A, F, s, iters=800):
    from catboost import CatBoostRanker, Pool
    from catboost.utils import get_gpu_device_count
    dev = "GPU" if get_gpu_device_count() > 0 else "CPU"
    pool = lambda X, y=None, g=None: Pool(X.assign(task_index=X.task_index.astype(int)), y, group_id=g,
                                          cat_features=["task_index"])
    m = CatBoostRanker(loss_function="YetiRank", iterations=iters, learning_rate=0.08, depth=6, random_seed=s,
                       task_type=dev, verbose=0)
    m.fit(pool(A[F], A.target_index, A.sample_id.values))
    return lambda X: m.predict(pool(X))


def ent_xgb(A, F, s):
    import xgboost as xgb
    m = xgb.XGBRanker(objective="rank:pairwise", n_estimators=400, learning_rate=0.05, max_depth=6, subsample=0.8,
                      colsample_bytree=0.7, tree_method="hist", device="cuda", random_state=s)
    m.fit(A[F], A.target_index, qid=A.sample_id.values)
    return m.predict

In [ ]:
cv_modelo("lgb_bin", ent_lgb_bin, GRUPOS_ACTUALES, nota="LightGBM binario")

In [ ]:
cv_modelo("xgb_rank", ent_xgb, GRUPOS_ACTUALES, nota="XGBoost rank:pairwise")

In [ ]:
cv_modelo("cat_rank", ent_cat, GRUPOS_ACTUALES, nota="CatBoost YetiRank")

**Conclusión bloque 8 · (rellenar tras ejecutar)**

| Modelo | CV | Δ vs `lgb_adri` (0.656) | peor controlador |
|---|---|---|---|
| … | | | |

## 9 · ¿Suman al stacking?

**Pregunta:** aunque un modelo saque menos solo, ¿mejora al mezclarlo con `lgb_adri`? Se mezclan los **rankings dentro del snapshot**
(percentil de cada tarea) con varios pesos, como hace `stacking/stack.py`. Si alguno sube la mezcla por encima de 0.656, merece
guardarse para Juanma (celda de abajo). Los pesos se eligen mirando el mismo OOF: es una estimación optimista; la de verdad la
da `stack.py` con CV anidada. No entrena nada, ~30 s por modelo.

In [ ]:
def mezcla_con_lgb(nombre, pesos=(0, 0.25, 0.5, 0.75, 1)):
    d, oof, _ = OOFS[nombre]
    x = d[["sample_id", "row_index", "target_index"]].assign(nuevo=oof).merge(
        pd.read_csv(ROOT / "oof" / f"{NAME}.csv").rename(columns={"score": "lgb"}), on=["sample_id", "row_index"])
    for c in ("nuevo", "lgb"):
        x[c] = x.groupby("sample_id")[c].rank(pct=True)
    h_lgb, h_nuevo = hit3(x, x.lgb), hit3(x, x.nuevo)
    print(f"{nombre}: aciertan los dos {(h_lgb & h_nuevo).mean():.3f} | solo lgb_adri {(h_lgb & ~h_nuevo.astype(bool)).mean():.3f}"
          f" | solo {nombre} {(~h_lgb.astype(bool) & h_nuevo).mean():.3f}")
    return pd.Series({w: hit3(x, (1 - w) * x.lgb + w * x.nuevo).mean() for w in pesos},
                     name=f"peso de {nombre}").round(4)


pd.concat([mezcla_con_lgb(m) for m in ["lgb_bin", "xgb_rank", "cat_rank"] if m in OOFS], axis=1)

In [ ]:
# Guardar para el stacking de Juanma SOLO los modelos que suban la mezcla (entrena también con todo para test)
GUARDAR = []    # p. ej. [("cat_rank", ent_cat)]
for nombre, entrenar in GUARDAR:
    cv_modelo(nombre, entrenar, GRUPOS_ACTUALES, guardar=True, nota="versión para el stacking")

**Conclusión bloque 9 · (rellenar tras ejecutar)**

## 10 · Siguientes ideas

Del análisis del OOF de `lgb_adri` (sin entrenar), **qué merma la precisión**:

| | ≤ 25 tareas | 26–35 | 36–45 | > 45 |
|---|---|---|---|---|
| acierto | **0.76** | 0.69 | 0.63 | **0.58** |

| | top-1 | top-2 | top-3 | top-5 | top-10 |
|---|---|---|---|---|---|
| acierto | 0.355 | 0.546 | **0.656** | 0.779 | 0.904 |

- **Listas largas:** el 55 % de los snapshots tiene más de 35 tareas, y ahí el acierto cae a 0.58–0.63.
- **Fallos por poco:** 12 puntos entre top-3 y top-5; la respuesta está en el top-10 el 90 % de las veces → **bloque 11**.
- **Etiquetas con ruido:** un 10 % de las respuestas cae fuera del top-10 → **bloque 12**.
- **Controladores:** de 0.57 (controlador 16) a 0.78 (controlador 20). No se puede atacar sin ver otros snapshots de la sesión.
- Descartado: obligar a 3 aviones distintos en el top-3 empeora (0.647 frente a 0.656): cuando el modelo pone dos tareas del
  mismo avión, suele acertar.
- Pendiente: **ranking por avión en dos etapas** (tarea de Juanma, `m_two_stage.py`) y **versión robusta** con `regularizado`
  (bloque 4: +1.8 en el peor controlador a cambio de −0.7 de media).

## 11 · Re-ranker de segunda etapa sobre el top-10

**Hipótesis:** la respuesta está en el top-10 del primer modelo el 90 % de las veces, pero solo en el top-3 el 66 %. Un
**segundo LightGBM que solo ordena esos 10 candidatos** puede concentrarse en las comparaciones difíciles (y las listas largas
dejan de importar: siempre compara 10). Además de las features normales, ve **cómo puntuó el primer modelo a todo el snapshot**:

| Columna nueva | Qué es |
|---|---|
| `s1`, `s1_rk` | puntuación y posición en la etapa 1 (`lgb_adri`) |
| `p1`, `p1_top1`, `entropia` | probabilidad (softmax en el snapshot), la del favorito y cuánto duda el modelo |
| `s1_gap_top`, `s1_gap_3` | distancia al 1º y al 3º (¿está "en la frontera"?) |
| `ac_s1_gap`, `ac_es_mejor`, `ac_n` | frente a las otras tareas de su avión |
| `tipo_s1_rk`, `tipo_gap` | frente a las tareas de su mismo tipo |
| `c_n_tipo`, `c_n_aviones`, `c_n_<tipo>` | composición de los 10 candidatos (cuántos transfer, assume, zone…) |

**Cómo se mide sin trampa:** la etapa 1 son las predicciones OOF de `oof/lgb_adri.csv` (cada snapshot puntuado por un modelo que
no vio a su controlador). La etapa 2 se entrena y se evalúa con los **mismos folds**. Se evalúa sobre **todos** los snapshots:
si la respuesta no está en el top-K, cuenta como fallo. Para entrenar se quitan los snapshots sin respuesta en el top-K (no
aportan nada que ordenar).
*Matiz:* el OOF de un snapshot de entrenamiento lo generó un modelo que sí vio el fold de validación; es lo mismo que pasa en el
stacking de Juanma y el efecto es pequeño.

| Variante | Por qué |
|---|---|
| `k10` | top-10 (cubre el 90 %) con las features normales + contexto |
| `k15` | más candidatos (cubre más, pero compara más) |
| `k10_solo_ctx` | solo las columnas de contexto + `task_index`: ¿basta con lo que dice la etapa 1? |
| `k10_red` | top-10 + **la puntuación de la red de Pablo** (`nn_tf_v2`). Idea de Pablo: la red compara las tareas entre sí (atención) y falla en otros snapshots; aquí el árbol aprende **cuándo fiarse de ella** |
| `k10_red_types` | lo anterior + los especialistas por tipo de Juanma (`types`): más opiniones distintas sobre los mismos candidatos |

Por cada modelo extra se añaden 4 columnas: su puntuación (`<modelo>_s`), su posición en el snapshot (`_rk`), su probabilidad
(`_p`) y la distancia a su favorito (`_gap`). Salen de `oof/<modelo>.csv` (mismos folds) y, para test, de `preds/<modelo>.csv`.
Es un stacking "dentro del re-ranker": a diferencia de la mezcla del bloque 9, puede fiarse de la red **solo en algunos casos**.

2 semillas, ~1 min por variante (entrena sobre 10–15 filas por snapshot en vez de ~36).

In [ ]:
CTX = ["s1", "s1_rk", "p1", "p1_top1", "entropia", "s1_gap_top", "s1_gap_3", "ac_s1_gap", "ac_es_mejor", "ac_n",
       "tipo_s1_rk", "tipo_gap", "c_n_tipo", "c_n_aviones"] + [f"c_n_{t}" for t in (2, 3, 4, 5, 12, 13)]


def contexto_etapa1(d, s1, K, extra=None):
    """d: filas (lab o test) ordenadas por sample_id, row_index; s1: score de la etapa 1 alineado con d.
    extra: {modelo: scores alineados con d} de otros modelos (p. ej. la red).
    Devuelve solo los K mejores candidatos de cada snapshot, con las columnas CTX (+ las de los extra), mismo índice que d."""
    x = d[["sample_id", "cs", "task_index"]].assign(s1=np.asarray(s1))
    cols_extra = []
    for m, sc in (extra or {}).items():
        x[f"{m}_s"] = np.asarray(sc)
        gm = x.groupby("sample_id")[f"{m}_s"]
        x[f"{m}_rk"] = gm.rank(ascending=False, method="first")
        x[f"{m}_p"] = np.exp(x[f"{m}_s"] - gm.transform("max"))
        x[f"{m}_p"] /= x.groupby("sample_id")[f"{m}_p"].transform("sum")
        x[f"{m}_gap"] = x[f"{m}_s"] - gm.transform("max")
        cols_extra += [f"{m}_s", f"{m}_rk", f"{m}_p", f"{m}_gap"]
    g = x.groupby("sample_id").s1
    x["s1_rk"] = g.rank(ascending=False, method="first")
    x["p1"] = np.exp(x.s1 - g.transform("max"))
    x["p1"] /= x.groupby("sample_id").p1.transform("sum")
    x["p1_top1"] = x.groupby("sample_id").p1.transform("max")
    x["entropia"] = (-x.p1 * np.log(x.p1 + 1e-12)).groupby(x.sample_id).transform("sum")
    x["s1_gap_top"] = x.s1 - g.transform("max")
    x["s1_gap_3"] = x.s1 - x.sample_id.map(x[x.s1_rk == 3].set_index("sample_id").s1)
    ga = x.groupby(["sample_id", "cs"]).s1
    x["ac_s1_gap"] = x.s1 - ga.transform("max")
    x["ac_es_mejor"] = (x.ac_s1_gap == 0).astype(int)
    x["ac_n"] = ga.transform("size")
    gt = x.groupby(["sample_id", "task_index"]).s1
    x["tipo_s1_rk"] = gt.rank(ascending=False, method="first")
    x["tipo_gap"] = x.s1 - gt.transform("max")
    c = x[x.s1_rk <= K].copy()
    c["c_n_tipo"] = c.groupby(["sample_id", "task_index"]).s1.transform("size")
    c["c_n_aviones"] = c.groupby("sample_id").cs.transform("nunique")
    for t in (2, 3, 4, 5, 12, 13):
        c[f"c_n_{t}"] = (c.task_index == t).astype(int).groupby(c.sample_id).transform("sum")
    return d.loc[c.index].join(c[CTX + cols_extra]), cols_extra


def s1_de(df, carpeta, modelo=NAME):
    """Score de un modelo (por defecto la etapa 1, lgb_adri) alineado con df. carpeta: "oof" (lab) o "preds" (test)."""
    s = df[["sample_id", "row_index"]].merge(pd.read_csv(ROOT / carpeta / f"{modelo}.csv"),
                                             on=["sample_id", "row_index"], how="left").score
    assert s.notna().all(), f"{carpeta}/{modelo}.csv no cubre todas las filas"
    return s.values


def rerank(nombre, K=10, usar_F=True, seeds=SEEDS_BLOQUE, params=None, guardar=False, nota="", extra=()):
    t0 = time.time()
    params = {**PARAMS, **(params or {})}
    d, F = build(lab, MEJOR_GRUPOS)
    s1 = s1_de(d, "oof")
    c, cols_extra = contexto_etapa1(d, s1, K, {m: s1_de(d, "oof", m) for m in extra})
    FR = (F if usar_F else ["task_index"]) + CTX + cols_extra
    tiene = (c.groupby("sample_id").target_index.transform("max") == 1).values
    oof_c = np.zeros(len(c))
    for f in range(5):
        tr, va = ((c.fold != f).values & tiene), (c.fold == f).values
        for s in seeds:
            oof_c[va] += fit(c[tr], FR, {**params, "random_state": s}).predict(c.loc[va, FR]) / len(seeds)
    full = pd.Series(s1 - 1e6, index=d.index)          # fuera del top-K: por debajo, en el orden de la etapa 1
    full.loc[c.index] = oof_c
    h = hit3(d, full.values)
    peor = h.groupby(ATCO.reindex(h.index)).mean().min()
    cubre = c.groupby("sample_id").target_index.max().reindex(h.index).fillna(0).mean()
    fila = dict(hora=time.strftime("%H:%M"), nombre=nombre,
                grupos="+".join(MEJOR_GRUPOS) + f" | rerank top-{K}" + (f" + {'+'.join(extra)}" if extra else ""),
                params_cambiados=str({k: v for k, v in params.items() if PARAMS.get(k) != v}), n_features=len(FR),
                cv_top3=round(h.mean(), 4), std_folds=np.nan,
                nota=f"{len(seeds)} semillas. peor_atco {peor:.3f}. top-{K} cubre {cubre:.3f}. {nota}".strip())
    pd.DataFrame([fila]).to_csv(LOG, mode="a", header=not LOG.exists(), index=False)
    print(f"{nombre:28s} CV {h.mean():.4f}  peor_atco {peor:.3f}  (top-{K} contiene la respuesta: {cubre:.3f})"
          f"  {len(FR)} feats  {time.time() - t0:.0f}s", flush=True)
    OOFS[nombre] = (d, full.values, h)
    if guardar:
        dt, _ = build(test, MEJOR_GRUPOS)
        s1t = s1_de(dt, "preds")
        ct, _ = contexto_etapa1(dt, s1t, K, {m: s1_de(dt, "preds", m) for m in extra})
        st = np.mean([fit(c[tiene], FR, {**params, "random_state": s}).predict(ct[FR]) for s in seeds], axis=0)
        full_t = pd.Series(s1t - 1e6, index=dt.index)
        full_t.loc[ct.index] = st
        save_outputs(nombre, d[["sample_id", "row_index"]].assign(score=full.values),
                     dt[["sample_id", "row_index"]].assign(score=full_t.values))
    return h.mean(), peor

In [ ]:
res11 = [("etapa 1 (lgb_adri, 3 semillas)", 0.6562, 0.571)]
for nombre, K, usar_F, extra, por_que in [
        ("k10",           10, True,  (),                     "top-10 con features + contexto"),
        ("k15",           15, True,  (),                     "top-15 con features + contexto"),
        ("k10_solo_ctx",  10, False, (),                     "top-10 solo con el contexto de la etapa 1"),
        ("k10_red",       10, True,  ("nn_tf_v2",),          "top-10 + puntuación de la red de Pablo"),
        ("k10_red_types", 10, True,  ("nn_tf_v2", "types"),  "top-10 + red + especialistas por tipo")]:
    res11.append((nombre, *rerank(f"b11/{nombre}", K, usar_F, nota=por_que, extra=extra)))
t11 = pd.DataFrame(res11, columns=["variante", "cv", "peor_atco"])
t11["Δ_vs_etapa1"] = t11.cv - t11.cv.iloc[0]
t11.round(4)

**Conclusión bloque 11 · el re-ranker empeora en todas las variantes, también con la red de Pablo.**

| Variante (2 semillas) | CV | Δ vs etapa 1 (0.6562) | peor controlador | |
|---|---|---|---|---|
| k10_red_types | 0.6436 | −0.013 | 0.575 | empeora |
| k10 | 0.6357 | −0.021 | 0.565 | empeora |
| k15 | 0.6352 | −0.021 | 0.566 | empeora |
| k10_solo_ctx | 0.6346 | −0.022 | 0.549 | empeora |
| k10_red | 0.6319 | −0.024 | 0.559 | empeora |

- Explicación más probable: las puntuaciones de la etapa 1 vienen de **5 modelos distintos** (uno por fold), cada uno con su
  escala; el re-ranker aprende con las de 4 y se evalúa con las del 5º. Lo mismo que vio Pablo con su "2.º pase LightGBM"
  (sección 9 de `futuro/LGB_futuro.ipynb`): un segundo LightGBM sobre los scores del primero sobreajusta.
- Meter más opiniones (red + types) es lo que menos pierde, pero no llega a la etapa 1. **Se descarta.**

In [ ]:
# Versión para el stacking: SOLO si el re-ranker mejora. Escribe oof/, preds/ y submissions/ de lgb_rerank.
GUARDAR_RERANK = False
K_MEJOR, USAR_F_MEJOR, EXTRA_MEJOR = 10, True, ("nn_tf_v2",)     # <- la variante ganadora
if GUARDAR_RERANK:
    rerank("lgb_rerank", K_MEJOR, USAR_F_MEJOR, seeds=SEEDS_FINAL, guardar=True, nota="versión para el stacking",
           extra=EXTRA_MEJOR)

## 12 · Limpiar las etiquetas ruidosas al entrenar

**Hipótesis:** un 10 % de las respuestas cae fuera del top-10 del modelo. Muchas serán **anotaciones discutibles o estilos raros**
de un controlador; al entrenar, el modelo intenta aprenderlas igualmente y eso le mete ruido. Si se **quitan del entrenamiento**
los snapshots cuya respuesta el modelo pone muy lejos (posición > R en el OOF), aprende de los casos "razonables".
**Se evalúa siempre sobre todos los snapshots** (también los quitados), así que la nota no hace trampa.

| Variante | Qué se quita del entrenamiento |
|---|---|
| `R=25` | solo los casos extremos |
| `R=15` | intermedio |
| `R=10` | todo lo que cae fuera del top-10 (~10 % de los snapshots) |

*Matiz:* la posición sale del OOF de `lgb_adri`, y el OOF de un snapshot de entrenamiento lo generó un modelo que **sí vio** el
fold que se evalúa. El bloque 12b lo comprueba sin esa fuga. ~1 min por variante.

In [ ]:
def cv_limpio(nombre, R, seeds=SEEDS_BLOQUE, nota="", params_extra=None, peso_pot=None, d=None, F=None):
    """CV de MEJOR quitando del ENTRENAMIENTO los snapshots cuya respuesta está más allá de la posición R en el OOF
    de lgb_adri. params_extra: cambios de parámetros. peso_pot: peso 1/longitud_tramo^p en el entrenamiento (12c)."""
    t0 = time.time()
    params = {**PARAMS, **MEJOR_PARAMS, **(params_extra or {})}
    if d is None:
        d, F = build(lab, MEJOR_GRUPOS)
    rk = pd.Series(s1_de(d, "oof"), index=d.index).groupby(d.sample_id).rank(ascending=False, method="first")
    pos = rk[d.target_index == 1].groupby(d.sample_id[d.target_index == 1]).first()
    quitar = set(pos[pos > R].index) if R else set()
    usar = ~d.sample_id.isin(quitar).values
    w = None if peso_pot is None else 1.0 / longitud_tramo(d).values ** peso_pot
    oof = np.zeros(len(d))
    for f in range(5):
        tr, va = (d.fold != f).values & usar, (d.fold == f).values
        for s in seeds:
            oof[va] += fit(d[tr], F, {**params, "random_state": s}, None if w is None else w[tr]
                           ).predict(d.loc[va, F]) / len(seeds)
    h = hit3(d, oof)
    peor = h.groupby(ATCO.reindex(h.index)).mean().min()
    fila = dict(hora=time.strftime("%H:%M"), nombre=nombre, grupos="+".join(MEJOR_GRUPOS) or "-",
                params_cambiados=str({k: v for k, v in params.items() if PARAMS.get(k) != v}), n_features=len(F),
                cv_top3=round(h.mean(), 4), std_folds=np.nan,
                nota=f"{len(seeds)} semillas. peor_atco {peor:.3f}. sin {len(quitar)} snapshots al entrenar. "
                     f"{'' if peso_pot is None else f'peso tramo^{peso_pot}. '}{nota}".strip())
    pd.DataFrame([fila]).to_csv(LOG, mode="a", header=not LOG.exists(), index=False)
    print(f"{nombre:22s} CV {h.mean():.4f}  peor_atco {peor:.3f}  (quitados al entrenar: {len(quitar)}, "
          f"{len(quitar) / d.sample_id.nunique():.1%})  {time.time() - t0:.0f}s", flush=True)
    OOFS[nombre] = (d, oof, h)
    return h.mean(), peor, len(quitar)


def longitud_tramo(d):
    """Por fila: nº de snapshots seguidos de la sesión (hueco ≤ 4.5 s) con la misma respuesta. Usa la etiqueta y el
    timestamp SOLO para pesar el entrenamiento: la predicción de cada snapshot sigue usando solo sus filas."""
    y = d[d.target_index == 1][["sample_id", "session", "timestamp", "task_name"]].copy()
    y["t"] = pd.to_timedelta(y.timestamp).dt.total_seconds()
    y = y.sort_values(["session", "t"])
    nuevo = (y.session != y.session.shift()) | (y.t.diff() > 4.5) | (y.task_name != y.task_name.shift())
    seg = nuevo.cumsum()
    return d.sample_id.map(dict(zip(y.sample_id, seg.map(seg.value_counts()))))


if REPETIR_HECHOS:
    res12 = [(nombre, *cv_limpio(f"b12/{nombre}", R, nota=por_que)) for nombre, R, por_que in [
        ("sin_limpiar", None, "referencia"), ("R=25", 25, "quitar casos extremos"),
        ("R=15", 15, "quitar intermedio"), ("R=10", 10, "quitar todo lo de fuera del top-10")]]
    t12 = pd.DataFrame(res12, columns=["variante", "cv", "peor_atco", "quitados"])
    t12["Δ_vs_sin_limpiar"] = t12.cv - t12.cv.iloc[0]
    display(t12.round(4))
else:
    print("Bloque ya ejecutado: ver la conclusión. REPETIR_HECHOS = True para repetirlo.")

**Conclusión bloque 12 · limpiar ayuda: la mejor nota del LightGBM limpio hasta ahora (pendiente de confirmar en 12b).**

| Variante (2 semillas) | CV | Δ vs sin limpiar (0.6569) | peor controlador | snapshots quitados al entrenar |
|---|---|---|---|---|
| **R=10** | **0.6649** | **+0.008** | 0.560 | 1.266 (9,6 %) |
| R=15 | 0.6622 | +0.005 | **0.576** | 613 (4,7 %) |
| R=25 | 0.6554 | −0.002 | 0.566 | 81 (0,6 %) |

- Quitar del entrenamiento lo que el modelo pone fuera del top-10 sube 0.8 puntos; R=15 sube algo menos pero **mejora el peor
  controlador**. Quitar solo los casos extremos (R=25) no hace nada.
- **Por encima del stacking de Juanma** (0.660) con un solo modelo.
- Hay una pequeña fuga en cómo se eligen los snapshots a quitar (ver el matiz de arriba): **12b** lo mide sin ella.

### 12b · Comprobación anidada (sin fuga) y robustez por sesión y escenario

**Qué cambia:** para cada fold externo `f`, las posiciones que deciden qué quitar se calculan **solo con los otros 4 folds**
(un CV interno: se entrena con 3 y se puntúa el 4º). Así el filtro nunca ha visto el fold `f`, que es el que se evalúa.

**Robustez (lección de Pablo, sección 12d de `futuro/LGB_futuro.ipynb`):** con folds por controlador, la media de la CV la
mueven unas pocas sesiones. Además del CV y del peor controlador se mira **cuántas sesiones suben o bajan, la mediana del
cambio por sesión y el cambio por escenario**. El privado (023) tiene los 3 escenarios.

~6 min (20 modelos internos + 2 semillas × 3 variantes × 5 folds).

In [ ]:
SES = lab.drop_duplicates("sample_id").set_index("sample_id")[["session", "scenario"]]


def robustez(h_ref, h_nuevo, nombre=""):
    """Compara dos series de aciertos por muestra: por sesión y por escenario."""
    x = SES.loc[h_ref.index].assign(ref=h_ref.values, nuevo=h_nuevo.reindex(h_ref.index).values)
    s = x.groupby("session").agg(scenario=("scenario", "first"), ref=("ref", "mean"), nuevo=("nuevo", "mean"))
    s["Δ"] = s.nuevo - s.ref
    print(f"{nombre}: Δ media {x.nuevo.mean() - x.ref.mean():+.4f} | sesiones que suben {(s.Δ > 0).sum()}, "
          f"bajan {(s.Δ < 0).sum()}, iguales {(s.Δ == 0).sum()} | Δ mediana por sesión {s.Δ.median():+.4f}")
    e = x.groupby("scenario")[["ref", "nuevo"]].mean()
    e["Δ"] = e.nuevo - e.ref
    e["sesiones_suben/bajan"] = s.groupby("scenario").Δ.apply(lambda v: f"{(v > 0).sum()} / {(v < 0).sum()}")
    e["Δ_mediana_sesión"] = s.groupby("scenario").Δ.median()
    return e.round(4)


def posiciones_anidadas(d, F, params, f):
    """Posición de la respuesta de los snapshots de entrenamiento del fold externo f, calculada SOLO con los otros folds
    de entrenamiento (CV interno). Nunca usa el fold f."""
    folds_tr = [g for g in range(5) if g != f]
    pos = []
    for g in folds_tr:
        A, B = d[d.fold.isin([x for x in folds_tr if x != g])], d[d.fold == g]
        rk = pd.Series(fit(A, F, {**params, "random_state": 0}).predict(B[F]), index=B.index
                       ).groupby(B.sample_id).rank(ascending=False, method="first")
        t = (B.target_index == 1).values
        pos.append(pd.Series(rk[t].values, index=B.sample_id[t].values))
    return pd.concat(pos)


def cv_limpio_anidado(Rs=(None, 15, 10), seeds=SEEDS_BLOQUE):
    t0 = time.time()
    params = {**PARAMS, **MEJOR_PARAMS}
    d, F = build(lab, MEJOR_GRUPOS)
    oofs, quitados = {R: np.zeros(len(d)) for R in Rs}, {R: 0 for R in Rs}
    for f in range(5):
        pos = posiciones_anidadas(d, F, params, f)
        va = (d.fold == f).values
        for R in Rs:
            quitar = set(pos[pos > R].index) if R else set()
            quitados[R] += len(quitar)
            tr = (d.fold != f).values & ~d.sample_id.isin(quitar).values
            for s in seeds:
                oofs[R][va] += fit(d[tr], F, {**params, "random_state": s}).predict(d.loc[va, F]) / len(seeds)
        print(f"  fold {f} hecho ({time.time() - t0:.0f}s)", flush=True)
    filas = []
    for R in Rs:
        h = hit3(d, oofs[R])
        nombre = f"b12b/R={R}" if R else "b12b/sin_limpiar"
        OOFS[nombre] = (d, oofs[R], h)
        peor = h.groupby(ATCO.reindex(h.index)).mean().min()
        pd.DataFrame([dict(hora=time.strftime("%H:%M"), nombre=nombre, grupos="+".join(MEJOR_GRUPOS),
                           params_cambiados=str(MEJOR_PARAMS), n_features=len(F), cv_top3=round(h.mean(), 4),
                           std_folds=np.nan, nota=f"{len(seeds)} semillas. peor_atco {peor:.3f}. filtro ANIDADO, "
                                                  f"{quitados[R] / 4:.0f} quitados por fold de media")]
                     ).to_csv(LOG, mode="a", header=False, index=False)
        filas.append(dict(variante=nombre.split("/")[1], cv=h.mean(), peor_atco=peor))
    t = pd.DataFrame(filas)
    t["Δ_vs_sin_limpiar"] = t.cv - t.cv.iloc[0]
    return t.round(4)


tabla12b = cv_limpio_anidado()
display(tabla12b)
for R in (15, 10):
    display(robustez(OOFS["b12b/sin_limpiar"][2], OOFS[f"b12b/R={R}"][2], f"R={R} (anidado)"))

**Conclusión bloque 12b · (rellenar tras ejecutar)**

| Variante (filtro anidado) | CV | Δ vs sin limpiar | peor controlador | sesiones suben / bajan | Δ mediana por sesión |
|---|---|---|---|---|---|
| … | | | | | |

- Si la mejora se mantiene (aunque sea algo menor que en el bloque 12): **es real** → bloques 12c y 12d.
- Si desaparece: era la fuga; no se usa el filtro.

### 12c · Ideas del notebook de futuro de Pablo, sobre la versión limpia

`futuro/LGB_futuro.ipynb` probó muchas cosas con el futuro. Casi todas usan otros snapshots (segundo paso de sesión,
suavizado, oráculo) y **no valen aquí** (regla 1). Las que sí se pueden traer, porque solo cambian **cómo se entrena** y
la predicción de cada snapshot sigue usando solo sus filas:

| Variante | De dónde sale | Qué cambia | Por qué podría ayudar aquí |
|---|---|---|---|
| `l63_mc100` | sección 12a (con futuro: +0.4 con 1 semilla, no confirmado) | `num_leaves=63`, `min_child_samples=100` | árboles más ricos pero con hojas grandes: más interacciones sin memorizar |
| `peso_tramo` | sección 11e (con futuro: ruido) | cada snapshot pesa 1/√(longitud de su tramo) | los tramos largos (muchos snapshots casi iguales) dominan el entrenamiento; con etiquetas ruidosas, que un tramo raro y largo pese menos puede ayudar más que con futuro |
| **10 semillas** | sección 9b (+0.15 de forma robusta) | media de 10 modelos | más estable; se usa directamente en la versión final (12d), no hace falta probarlo |

Se prueban **sobre el filtro ganador del bloque 12** (`R_LIMPIO`), con el filtro simple (comparación justa entre variantes),
2 semillas, ~1–2 min por variante.

In [ ]:
R_LIMPIO = 10        # <- el R que gane en 12b (10 o 15)
d12, F12 = build(lab, MEJOR_GRUPOS)
res12c = [(nombre, *cv_limpio(f"b12c/{nombre}", R_LIMPIO, nota=por_que, params_extra=p, peso_pot=w, d=d12, F=F12))
          for nombre, p, w, por_que in [
              ("base", None, None, "referencia: filtro R sin más"),
              ("l63_mc100", dict(num_leaves=63, min_child_samples=100), None, "idea Pablo 12a"),
              ("peso_tramo", None, 0.5, "idea Pablo 11e: 1/sqrt(longitud del tramo)"),
          ]]
t12c = pd.DataFrame(res12c, columns=["variante", "cv", "peor_atco", "quitados"])
t12c["Δ_vs_base"] = t12c.cv - t12c.cv.iloc[0]
t12c.round(4)

**Conclusión bloque 12c · (rellenar tras ejecutar)**

| Variante | CV | Δ vs base del bloque | peor controlador | |
|---|---|---|---|---|
| … | | | | |

### 12d · Versión final limpia para el stacking y Kaggle: `lgb_adri_limpio`

`MEJOR` + filtro `R_LIMPIO` + lo que gane en 12c, con **10 semillas** (idea de Pablo). Guarda en el formato común:
`oof/lgb_adri_limpio.csv`, `preds/lgb_adri_limpio.csv` y `submissions/sub_lgb_adri_limpio.csv` (no pisa `lgb_adri`).
Para test se entrena con todo train+validation **menos** los snapshots filtrados. Al final compara con `lgb_adri` por sesión y
escenario. ~8–10 min. **Solo con `GENERAR_LIMPIO = True`.**

In [ ]:
GENERAR_LIMPIO = False
PARAMS_LIMPIO = {}           # <- p. ej. dict(num_leaves=63, min_child_samples=100) si gana en 12c
PESO_LIMPIO = None           # <- 0.5 si peso_tramo gana en 12c
SEEDS_LIMPIO = list(range(10))

if GENERAR_LIMPIO:
    nombre = "lgb_adri_limpio"
    cv_l, peor_l, n_q = cv_limpio(nombre, R_LIMPIO, seeds=SEEDS_LIMPIO, params_extra=PARAMS_LIMPIO, peso_pot=PESO_LIMPIO,
                                  nota="versión para el stacking", d=d12, F=F12)
    d, oof, h = OOFS[nombre]
    params = {**PARAMS, **MEJOR_PARAMS, **PARAMS_LIMPIO}
    rk = pd.Series(s1_de(d, "oof"), index=d.index).groupby(d.sample_id).rank(ascending=False, method="first")
    pos = rk[d.target_index == 1].groupby(d.sample_id[d.target_index == 1]).first()
    usar = ~d.sample_id.isin(set(pos[pos > R_LIMPIO].index)).values
    w = None if PESO_LIMPIO is None else 1.0 / longitud_tramo(d).values[usar] ** PESO_LIMPIO
    dt, _ = build(test, MEJOR_GRUPOS)
    score = np.mean([fit(d[usar], F12, {**params, "random_state": s}, w).predict(dt[F12]) for s in SEEDS_LIMPIO], axis=0)
    save_outputs(nombre, d[["sample_id", "row_index"]].assign(score=oof), dt[["sample_id", "row_index"]].assign(score=score))
    json.dump(dict(groups=MEJOR_GRUPOS, params=params, R=R_LIMPIO, peso_pot=PESO_LIMPIO, seeds=SEEDS_LIMPIO,
                   cv_top3=round(cv_l, 4)), open(ROOT / "models" / f"{nombre}_config.json", "w"), indent=1)
    ref = lab.merge(pd.read_csv(ROOT / "oof" / f"{NAME}.csv"), on=["sample_id", "row_index"])
    display(robustez(hit3(ref, ref.score), h, f"{nombre} frente a lgb_adri"))
else:
    print("No se genera (GENERAR_LIMPIO = False).")

**Resultado bloque 12d · (rellenar tras ejecutar)**

| | CV top-3 | peor controlador | sesiones suben / bajan | Kaggle público |
|---|---|---|---|---|
| `lgb_adri` | 0.656 | 0.571 | — | |
| **`lgb_adri_limpio`** | | | | |

- Para Juanma: `oof/lgb_adri_limpio.csv` y `preds/lgb_adri_limpio.csv` entran solos en `stacking/stack.py`.
- Para Kaggle (lo sube Juanma): `submissions/sub_lgb_adri_limpio.csv`.

## Historial

In [ ]:
historial(30)